In [58]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [59]:
from daemon_analysis_tools.io.csv_handler import load_and_process_csv
from daemon_analysis_tools.io.yaml_handler import save_answers_to_yaml, load_answers_from_yaml
from daemon_analysis_tools.processing.grouper import group_questions_by_journal
from daemon_analysis_tools.services.discrepancy_resolver import resolve_discrepancy

In [60]:
! pwd

/home/lukas/Science/Projects/DAEMON_open_data_policies/Journal-Research-Data-Policy/notebooks/fix_inconsistencies


Load and process data:
- Group answers by publisher and journal, trying to uniform names written in slightly different ways.
- Store in a DataFrame

In [61]:
data = load_and_process_csv("../../data/raw/rdp.csv")

Get a `dict` labeled by publisher names of `dict`s labeled by journal names of `dict`s of `Question` instances. The `.answer` attribute contains the answers given by the respondents and the explanations text to motivate it.

In [62]:
question_metadata_file = "../../data/metadata/question_metadata.yaml"

grouped_questions = group_questions_by_journal(data, question_metadata_file)

## Resolve discrepancies

The `Question` class has a `.resolve_discrepancies` method which updates `Question.anwsers` with the correct answer.

For example, let's consider IOP's 2D Materials. Question 7 has discrepancies.

In [63]:
for journal, data in grouped_questions["AIP"].items():
    print("#############################################################")
    print(journal)
    for question, answer in data.items():
        if answer.has_discrepancies():
            answer.print_qa()

#############################################################

11. Unique policies for data types
Please name all data types for which unique policies are recommended according to the RDP (Please separate all data types be semicolon, e.g. crystal structures; Protein sequence)
  Resp. 0:
    Answer: none
    Explanation: 
12. Unique policies for data types
Please name all data types for which unique policies are required according to the RDP (Please separate all data types be semicolon, e.g. crystal structures; Protein sequence)
  Resp. 0:
    Answer: none
    Explanation: 
#############################################################
aip_advances
3. Data sharing requirements in RDP
  Resp. 0:
    Answer: Data sharing required but not publicly (e.g. available upon request is allowed).
    Explanation: Below is a list of standard templates for the text that will appear in the “Data Availability Statement” portion of your article.  When multiple data sets are being described, please provi

Inconsistencies can be removed manually, passing the index of the correct respondent.

In [64]:
for j in ["aip_advances", "apl_materials", "apl_photonics", "applied_physics_letters",
          "applied_physics_reviews", "chinese_journal_of_chemical_physics",
          "journal_of_mathematical_physics", "low_temperature_physics", ]:
    
    for i in [3]:
        resolve_discrepancy(
            grouped_questions["AIP"][j][i],
            correct_answer=1,
            discrepancy_reason="Language understanding",
        )
    

for j in ["journal_of_applied_physics", "physics_of_fluids", "physics_of_plasmas",
          "review_of_scientific_instruments", ]:
      
    for i in [3]:
        resolve_discrepancy(
            grouped_questions["AIP"][j][i],
            correct_answer=1,
            discrepancy_reason="Language understanding",
        )

    for i in [8]:
        resolve_discrepancy(
            grouped_questions["AIP"][j][i],
            correct_answer=0,
            discrepancy_reason="Text not found",
        )


for j in ["structural_dynamics", ]:
      
    for i in [3]:
        resolve_discrepancy(
            grouped_questions["AIP"][j][i],
            correct_answer=1,
            discrepancy_reason="Language understanding",
        )

    for i in [7, 10]:
        resolve_discrepancy(
            grouped_questions["AIP"][j][i],
            correct_answer=1,
            discrepancy_reason="Text not found",
        )


for j in ["the_journal_of_chemical_physics", ]:
      
    for i in [3]:
        resolve_discrepancy(
            grouped_questions["AIP"][j][i],
            correct_answer=1,
            discrepancy_reason="Language understanding",
        )

    for i in [10]:
        resolve_discrepancy(
            grouped_questions["AIP"][j][i],
            correct_answer=1,
            discrepancy_reason="Text not found",
        )

    for i in [13, 14]:
        resolve_discrepancy(
            grouped_questions["AIP"][j][i],
            correct_answer=0,
            discrepancy_reason="Language understanding",
        )



In [65]:
for journal, data in grouped_questions["AIP"].items():
    print("#############################################################")
    print(journal)
    for question, answer in data.items():
        if answer.has_discrepancies() and answer.correct_answer is None:
            answer.print_qa()

#############################################################

11. Unique policies for data types
Please name all data types for which unique policies are recommended according to the RDP (Please separate all data types be semicolon, e.g. crystal structures; Protein sequence)
  Resp. 0:
    Answer: none
    Explanation: 
12. Unique policies for data types
Please name all data types for which unique policies are required according to the RDP (Please separate all data types be semicolon, e.g. crystal structures; Protein sequence)
  Resp. 0:
    Answer: none
    Explanation: 
#############################################################
aip_advances
#############################################################
apl_materials
#############################################################
apl_photonics
#############################################################
applied_physics_letters
#############################################################
applied_physics_reviews
#####################

In [66]:
save_answers_to_yaml(
    grouped_questions,
    parent_folder="../../data/processed/all_answers",
    save_only=["AIP"],
)

After doing this, the `.get_final_answer()` method returns the correct answer.